⏱️ **Time required:** ~2 minutes | **Type:** Interactive dashboards

# RideFlow Interactive Dashboards

This notebook visualizes the Data Mesh using Panel directly from your Lakehouse output. Execute the cells sequentially to spin up interactive dashboards directly within your notebook environment!

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

In [ ]:
# hvplot
# jupyter_bokeh
#

In [3]:
import panel as pn
import os

# Ensure our internal dashboards folder can be resolved
import sys

sys.path.append(os.path.abspath("."))

# Initialize Panel styling for notebooks
pn.extension("tabulator", design="bootstrap")

## 1. Executive Mesh View

A cross-domain rollup of pipeline health metrics and top-level KPIs.

In [1]:
import panel as pn
import polars as pl
import hvplot.pandas
import os


# Create mock data or load real delta data for Executive View
def create_executive_view():
    # Premium styled KPI Cards using custom HTML
    card_style = """
    <div style="background-color: #1e1e1e; border-radius: 8px; padding: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.3); text-align: center; border-top: 4px solid {color};">
        <div style="color: #a0a0a0; font-size: 14px; text-transform: uppercase; letter-spacing: 1px; margin-bottom: 10px;">{title}</div>
        <div style="color: #ffffff; font-size: 32px; font-weight: bold; font-family: 'Inter', sans-serif;">{value}</div>
    </div>
    """

    kpi_cards = pn.Row(
        pn.pane.HTML(
            card_style.format(title="GMV Today", value="2.4M £", color="#2ecc71"), sizing_mode="stretch_width"
        ),
        pn.pane.HTML(
            card_style.format(title="Active Riders", value="48,200", color="#3498db"), sizing_mode="stretch_width"
        ),
        pn.pane.HTML(
            card_style.format(title="Active Drivers", value="12,100", color="#e67e22"), sizing_mode="stretch_width"
        ),
        pn.pane.HTML(card_style.format(title="Take Rate", value="28.5%", color="#9b59b6"), sizing_mode="stretch_width"),
        sizing_mode="stretch_width",
        margin=(0, 0, 30, 0),
    )

    # Domain Health Matrix
    health_data = [
        {
            "Pipeline Area": "Trips",
            "Status": "✅ GREEN",
            "Freshness": "< 2 min",
            "Quality": "99.8%",
            "Quarantine": "0.2%",
        },
        {
            "Pipeline Area": "Riders",
            "Status": "✅ GREEN",
            "Freshness": "< 5 min",
            "Quality": "99.1%",
            "Quarantine": "0.9%",
        },
        {
            "Pipeline Area": "Drivers",
            "Status": "✅ GREEN",
            "Freshness": "< 5 min",
            "Quality": "98.5%",
            "Quarantine": "1.5%",
        },
        {
            "Pipeline Area": "Telemetry",
            "Status": "⚠️ AMBER",
            "Freshness": "22 min",
            "Quality": "94.2%",
            "Quarantine": "5.8%",
        },
    ]

    df_health = pl.DataFrame(health_data).to_pandas()

    health_title = pn.pane.Markdown("### 🏢 Domain Health Matrix", margin=(20, 0, 0, 0))
    health_table = pn.widgets.Tabulator(
        df_health, layout="fit_data_stretch", theme="fast", show_index=False, disabled=True
    )

    # Cost & Pipeline Stats
    cost_data = [
        {"Platform": "Google Ads (CAC)", "Cost": "£8.40"},
        {"Platform": "Meta Ads (CAC)", "Cost": "£14.20"},
        {"Platform": "Pipeline Compute (Per Run)", "Cost": "£4.12"},
    ]
    df_cost = pl.DataFrame(cost_data).to_pandas()

    cost_table = pn.widgets.Tabulator(df_cost, layout="fit_data_stretch", theme="fast", show_index=False, disabled=True)

    body = pn.Column(
        kpi_cards,
        pn.Row(
            pn.Column(health_title, health_table, sizing_mode="stretch_width"),
            pn.Column(
                pn.pane.Markdown("### 💰 Financials & Compute", margin=(20, 0, 0, 0)),
                cost_table,
                sizing_mode="stretch_width",
            ),
        ),
        sizing_mode="stretch_both",
    )

    return body


view_1 = create_executive_view()
view_1

## 2. Pipeline Observatory

Deep dive into LakeLogic's system-generated `lakehouse/_logs/` Delta tables monitoring Quarantine hits, processing SLAs, and computation metadata.

In [5]:
import panel as pn
import os

pn.extension("tabulator")


def create_observatory_view():
    log_base = r"./lakehouse"

    # Try reading the actual delta logs from the platform
    all_logs = []

    # Attempt to pull logs from all domains
    if os.path.exists(log_base):
        for domain in ["marketplace"]:
            dpath = os.path.join(log_base, domain, "_logs")
            if os.path.exists(dpath) and os.path.exists(os.path.join(dpath, "_delta_log")):
                try:
                    df = pl.read_delta(dpath)
                    all_logs.append(df)
                except Exception as e:
                    print(f"Error reading {domain} log: {e}")

    if all_logs:
        df_logs = pl.concat(all_logs)

        # Calculate summary metrics per domain
        # Example metrics: total_runs, total_duration, cost_usd, rows_quarantined

        df_summary = df_logs.group_by("domain", "system", "layer").agg(
            [
                pl.col("run_id").count().alias("runs"),
                pl.col("target_records").sum().alias("total_records"),
                pl.col("quarantined_records").sum().alias("quarantine_hits"),
                pl.col("duration_ms").mean().alias("avg_duration_ms"),
                pl.col("cost_usd").sum().alias("total_cost_usd"),
            ]
        )

        # Turn to pandas for hvplot
        pdf = df_summary.to_pandas()

        # Convert the raw logs for the datatable
        pdf_logs = (
            df_logs.select(
                [
                    "started_at",
                    "domain",
                    "system",
                    "entity",
                    "layer",
                    "status",
                    "target_records",
                    "quarantined_records",
                    "cost_usd",
                ]
            )
            .sort("started_at", descending=True)
            .head(50)
            .to_pandas()
        )

        # KPI Bar Chart
        bar_chart = pdf.hvplot.bar(
            x="domain",
            y="quarantine_hits",
            by="layer",
            stacked=True,
            title="Quarantine Hits by Layer per Domain",
            ylabel="Quarantined Records",
            height=300,
        )

        datatable = pn.widgets.Tabulator(
            pdf_logs, layout="fit_data_stretch", pagination="remote", page_size=15, theme="fast"
        )

        body = pn.Column(
            pn.pane.Markdown("### 🔍 Live Pipeline SLA Diagnostics"),
            bar_chart,
            pn.pane.Markdown("#### Recent Execution Logs", margin=(20, 0, 0, 0)),
            datatable,
            sizing_mode="stretch_both",
        )

        return body
    else:
        return pn.pane.Markdown("No Delta run logs exist in `lakehouse/_logs/`. Please run the pipeline first!")


view_2 = create_observatory_view()
view_2

## 3. Marketplace Pulse

Visualize downstream PII-masked operational views tapping natively onto Silver data.

In [6]:
import panel as pn
import os

pn.extension("tabulator")


def create_trip_operations_view():
    trips_path = r"./lakehouse\marketplace\silver\silver_rideflow_trips"

    if os.path.exists(trips_path):
        try:
            df_trips = pl.read_delta(trips_path)

            # Show a subset of trips
            df_display = df_trips.head(100).to_pandas()

            # Simple aggregation by driver for chart
            df_driver_counts = (
                df_trips.group_by("driver_id")
                .agg(pl.count("trip_id").alias("trip_count"))
                .sort("trip_count", descending=True)
                .head(15)
                .to_pandas()
            )

            bar_chart = df_driver_counts.hvplot.bar(
                x="driver_id", y="trip_count", title="Top 15 Drivers by Completed Trips", rot=45, height=300
            )

            datatable = pn.widgets.Tabulator(
                df_display, layout="fit_data_stretch", pagination="remote", page_size=12, theme="fast"
            )

            body = pn.Column(
                pn.pane.Markdown("### 🚗 Marketplace Pulse: Silver Trips Component"),
                bar_chart,
                pn.pane.Markdown("#### PII Masked Silver Trip Sample Feed", margin=(20, 0, 0, 0)),
                datatable,
                sizing_mode="stretch_both",
            )
            return body

        except Exception as e:
            return pn.pane.Markdown(f"### Error reading trips delta lake: {e}")

    return pn.pane.Markdown("### `marketplace.silver_rideflow_trips` table has not been generated.")


view_3 = create_trip_operations_view()
view_3

## 4. Compliance: EU AI Act Model Registry

Simulates how Data Contracts inherently track ML model governance natively in the Lakehouse.

In [ ]:
import panel as pn
import os
import yaml
import glob

pn.extension("tabulator")


def load_ai_act_contracts(base_path):
    registry_data = []

    # Search for all contracts across domains
    search_pattern = os.path.join(base_path, "**", "*.yaml")
    contract_files = glob.glob(search_pattern, recursive=True)

    for cfile in contract_files:
        try:
            with open(cfile, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)

            # Check if this contract has EU AI Act metadata
            compliance = data.get("compliance", {})
            ai_act = compliance.get("eu_ai_act", {})

            if ai_act and ai_act.get("applicable", False):
                registry_data.append(
                    {
                        "Model Domain": data.get("domain", "Unknown"),
                        "System": data.get("system", "Unknown"),
                        "Entity": data.get("dataset", "Unknown"),
                        "Risk Tier": str(ai_act.get("risk_tier", "Unknown")).upper(),
                        "Purpose": ai_act.get("ai_system_purpose", ""),
                        "Bias Examination": "✅" if ai_act.get("bias_examination", False) else "❌",
                        "Transparency": "✅" if ai_act.get("transparency_disclosure", False) else "❌",
                        "Human Oversight": "✅" if ai_act.get("human_oversight", False) else "❌",
                        "Logging": "✅" if ai_act.get("logging_enabled", False) else "❌",
                    }
                )
        except Exception:
            pass

    return registry_data


def create_ai_registry_view():
    contract_base = r"./assets/domains_rideflow"
    registry_data = load_ai_act_contracts(contract_base)

    if not registry_data:
        return pn.pane.Markdown("### No AI Act compliance definitions found in the contracts.")

    df = pl.DataFrame(registry_data).to_pandas()

    # Render table
    datatable = pn.widgets.Tabulator(df, layout="fit_data_stretch", theme="fast", show_index=False)

    # Count metrics
    high_risk = sum(1 for d in registry_data if d["Risk Tier"] == "HIGH")
    limited_risk = sum(1 for d in registry_data if d["Risk Tier"] == "LIMITED")

    kpi_cards = pn.Row(
        pn.indicators.Number(name="Total ML Models", value=len(registry_data), format="{value}", colors=[(1, "blue")]),
        pn.indicators.Number(
            name="High Risk Systems",
            value=high_risk,
            format="{value}",
            colors=[(1, "red")] if high_risk > 0 else [(1, "green")],
        ),
        pn.indicators.Number(name="Limited Risk Systems", value=limited_risk, format="{value}", colors=[(1, "orange")]),
        sizing_mode="stretch_width",
    )

    body = pn.Column(
        pn.pane.Markdown("### ⚖️ EU AI Act Compliance Registry (Regulation 2024/1689)"),
        pn.pane.Markdown(
            "This live registry auto-discovers all LakeLogic data contracts globally tagged with `eu_ai_act` metadata. It tracks bias-examination, human-oversight tooling, and risk categorization across the mesh."
        ),
        kpi_cards,
        pn.pane.Markdown("#### Registered AI/ML Pipeline Contracts", margin=(20, 0, 0, 0)),
        datatable,
        sizing_mode="stretch_both",
    )

    return body


view_4 = create_ai_registry_view()
view_4